In [46]:

!pip install langgraph
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain
!pip install langchain-community


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from typing import TypedDict

from langgraph.graph import StateGraph

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [48]:
# Define the state graph for the tutor system
class TutorState(TypedDict):

    topic: str

    context: str

    question: str

    student_answer: str

    score: float

    feedback: str

In [31]:
# load existing rag system
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [49]:
# Load vector store and embedding model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)
print("FAISS Loaded Successfully")

FAISS Loaded Successfully


In [50]:
# Model Evaluator
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Evaluator Model Loaded")

Evaluator Model Loaded


In [51]:
# Memory node
memory = {
    "weak_topics": [],
    "scores": []
}

In [52]:
# Retrieval node
def retrieval_node(state):

    docs = db.similarity_search(
        state["topic"],
        k=2
    )

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    print("\n===== RETRIEVED CONTEXT =====\n")

    print(context[:500])

    state["context"] = context

    return state

In [53]:
# Question Generation node
def question_generator_node(state):

    topic = state["topic"]

    if topic.lower() == "clustering":

        question = "What is clustering in machine learning and why is it used?"

    else:

        question = f"Explain the concept of {topic}."

    print("\n===== GENERATED QUESTION =====\n")

    print(question)

    state["question"] = question

    return state


In [55]:
# Evaluator node 
def evaluator_node(state):

    reference_answer = state["context"]

    student_answer = state["student_answer"]

    ref_embedding = model.encode([reference_answer])

    stu_embedding = model.encode([student_answer])

    similarity = cosine_similarity(
        ref_embedding,
        stu_embedding
    )[0][0]

    score = round(similarity * 10, 2)

    state["score"] = score

    return state

In [56]:
# Feedback node
def feedback_node(state):

    score = state["score"]

    if score >= 8:

        feedback = "Excellent understanding of clustering concepts."

    elif score >= 5:

        feedback = "Partial understanding detected. Add more technical details and algorithms."

    else:

        feedback = "Weak understanding detected. Revise clustering concepts carefully."

    state["feedback"] = feedback

    return state

In [57]:
# Memory node
def memory_node(state):

    memory["scores"].append(
        state["score"]
    )

    if state["score"] < 5:

        memory["weak_topics"].append(
            state["topic"]
        )

    return state

In [58]:
# Create the state graph
graph = StateGraph(TutorState)

In [ ]:
# add node
graph.add_node(
    "retrieval",
    retrieval_node
)

graph.add_node(
    "question_generator",
    question_generator_node
)

graph.add_node(
    "evaluator",
    evaluator_node
)

graph.add_node(
    "feedback",
    feedback_node
)

graph.add_node(
    "memory",
    memory_node
)

In [60]:
# Connect graph
graph.set_entry_point(
    "retrieval"
)

graph.add_edge(
    "retrieval",
    "question_generator"
)

graph.add_edge(
    "question_generator",
    "evaluator"
)

graph.add_edge(
    "evaluator",
    "feedback"
)

graph.add_edge(
    "feedback",
    "memory"
)

In [61]:
# Compile graph
app = graph.compile()

print(app)

In [62]:
# Run system
result = app.invoke({

    "topic": "clustering",

    "context": "",

    "question": "",

    "student_answer": """
Clustering is an unsupervised machine learning
technique used to group similar data points
based on similarities.
""",

    "score": 0,

    "feedback": ""
})


===== RETRIEVED CONTEXT =====

Clustering Algorithms
1 Introduction to Clustering
Clustering is anunsupervised learningtechnique that aims to group a set of data
objects into clusters such that objects within the same cluster are more similar to each
other than to those in other clusters. Similarity is usually measured using distance metrics
such as Euclidean, Manhattan, or cosine distance.
1.1 Objectives of Clustering
Clustering aims to organize unlabeled data into meaningful groups based on similarity
regularities by groupi

===== GENERATED QUESTION =====

What is clustering in machine learning and why is it used?


In [64]:
# Output results
print("\n===== QUESTION =====\n")

print(result["question"])

print("\n===== YOUR ANSWER =====\n")

print(result["student_answer"])

print("\n===== SCORE =====\n")

print(result["score"])

print("\n===== FEEDBACK =====\n")

print(result["feedback"])

print("\n===== MEMORY =====\n")

print(memory)



===== QUESTION =====

What is clustering in machine learning and why is it used?

===== YOUR ANSWER =====


Clustering is an unsupervised machine learning
technique used to group similar data points
based on similarities.


===== SCORE =====

8.26

===== FEEDBACK =====

Excellent understanding of clustering concepts.

===== MEMORY =====

{'weak_topics': [], 'scores': [np.float32(8.26)]}


In [42]:
# Add nodes and edges
graph.add_node(
    "retrieval",
    retrieval_node
)

graph.add_node(
    "question_generator",
    question_generator_node
)

graph.add_edge(
    "question_generator",
    "evaluator"
)
graph.add_node(
    "evaluator",
    evaluator_node
)

graph.add_node(
    "feedback",
    feedback_node
)

graph.add_node(
    "memory",
    memory_node
)

In [43]:
# Add edge

graph.set_entry_point(
    "retrieval"
)

graph.add_edge(
    "retrieval",
    "question_generator"
)

graph.add_edge(
    "question_generator",
    "student"
)

graph.add_edge(
    "student",
    "evaluator"
)

graph.add_edge(
    "evaluator",
    "feedback"
)

graph.add_edge(
    "feedback",
    "memory"
)

In [45]:
# Compile graph
app = graph.compile()
print(app)

ValueError: Found edge starting at unknown node 'student'

In [ ]:
# Run system
result = app.invoke({

    "topic": "clustering",

    "context": "",

    "question": "",

    "student_answer": "",

    "score": 0,

    "feedback": ""
})

In [ ]:
# Output results
print("\n===== QUESTION =====\n")

print(result["question"])

print("\n===== YOUR ANSWER =====\n")

print(result["student_answer"])

print("\n===== SCORE =====\n")

print(result["score"])

print("\n===== FEEDBACK =====\n")

print(result["feedback"])

print("\n===== MEMORY =====\n")

print(memory)


===== QUESTION =====


What is the main idea of the following context?

Clustering Algorithms
1 Introduction to Clustering
Clustering is anunsupervised learningtechnique that aims to group a set of data
objects into clusters such that objects within the same cluster are more similar to each
other than to those in other clusters. Similarity is usually measured using dist


===== YOUR ANSWER =====

Clustering is an unsupervised machine learning algorithm, which makes different groups with similar type of data points. K Means clustering is an example of this

===== SCORE =====

7.24

===== FEEDBACK =====

Partial understanding. Improve details.

===== MEMORY =====

{'weak_topics': [], 'scores': [np.float32(7.24)]}
